<div dir="rtl" lang="he">

# 🏠 ניתוח נתוני דירות להשכרה בתל אביב

פרויקט זה כולל שליפת נתונים (Web Scraping) מאתר [ad.co.il](https://www.ad.co.il), ממודעות השכרה בשלוש שכונות נבחרות בתל אביב:
**התקווה**, **בית שטראוס** ו־**הצפון הישן – החלק המרכזי**.

לאחר שליפת המידע, הנתונים עברו ניקוי, עיבוד והמרת טיפוסים, והוכנו לשימוש עבור ניתוחים או מודלים של למידת מכונה.

---


## 🛠 טכנולוגיות בשימוש

- `Python`
- `BeautifulSoup`, `requests` – לשליפת נתונים מהאתר
- `pandas` – לניקוי, עיבוד ושמירת הנתונים כ- Data Frame
- **Google Distance Matrix API** – לחישוב מרחק מכיכר דיזנגוף

---

## 📬 מוזמנים להשתמש, להציע שיפורים או פשוט להתרשם 🙂  

</div>


In [1]:
import requests
from bs4 import BeautifulSoup
import time

In [2]:
# מפת שכונות עם הקודים מאתר ad.co.il
area_codes = {
    "הצפון הישן - החלק המרכזי": 17553,
    "התקווה": 17586,
    "בית שטראוס": 17632
}


In [19]:
import requests

API_KEY = ""

def compute_distance(address):
    base_url = "https://maps.googleapis.com/maps/api/distancematrix/json"
    origin = f"{address}, תל אביב יפו"
    destination = "כיכר דיזנגוף, תל אביב"

    params = {
        "origins": origin,
        "destinations": destination,
        "key": API_KEY,
        "mode": "driving",
        "language": "he"
    }

    try:
        response = requests.get(base_url, params=params)
        result = response.json()
        distance = result["rows"][0]["elements"][0]["distance"]["value"]
        return distance
    except:
        return None


In [4]:
#  פונקציה שמביאה לינקים של דירות משכונה לפי הקוד שלה
def get_apartment_links_by_area(area_code, max_pages=3):
    base_url = "https://www.ad.co.il/nadlanrent"
    headers = {'User-Agent': 'Mozilla/5.0'}
    all_links = []

    for page in range(1, max_pages + 1):
        url = f"{base_url}?city=5000&sp277={area_code}&pageindex={page}"
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"שגיאה בדף {page} עבור קוד {area_code}")
            continue

        soup = BeautifulSoup(response.content, 'html.parser')
        cards = soup.find_all('div', class_='card-body')

        for card in cards:
            a_tag = card.find('a')
            if a_tag and a_tag.get('href'):
                full_url = "https://www.ad.co.il" + a_tag.get('href')
                all_links.append(full_url)

    return all_links


In [5]:
def get_apartment_info(link):
    apartment_dict = {}
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        response = requests.get(link, headers=headers)
        if response.status_code != 200:
            print(f"שגיאה בטעינת {link}")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')

        # שליפת נתונים מהטבלה
        table = soup.find('table', class_='table table-sm mb-4')
        if table:
            rows = table.find_all('tr')
            for row in rows:
                tds = row.find_all('td')
                if len(tds) == 2:
                    key = tds[0].get_text(strip=True)
                    value = tds[1].get_text(strip=True)
                    apartment_dict[key] = value

        # תיאור הדירה
        desc_container = soup.find('p', class_='text-word-break')
        apartment_dict['description'] = desc_container.get_text(strip=True) if desc_container else None

        # מספר תמונות בגלריה
        image_divs = soup.select('div.justify-content-center.px-1')
        apartment_dict['num_of_images'] = len(image_divs) if image_divs else nan

        # חישוב מרחק ממרכז העיר
        apt_address = apartment_dict.get("כתובת")
        apartment_dict["distance_from_center"] = compute_distance(apt_address)

        # מאפיינים בינאריים לפי אייקונים
        features_map = {
            'חניה': 'has_parking',
            'מחסן': 'has_stotsge',
            'מעלית': 'elevator',
            'מזגן': 'ac',
            'נגישות': 'handicap',
            'סורגים': 'has_bars',
            'ממ"ד': 'has_safe_room',
            'מרפסת': 'has_balcon',
            'מרוהטת': 'is_furnished',
            'משופצת': 'is_renovated'
        }

        for key in features_map.values():
            apartment_dict[key] = 0

        icons = soup.select('div.card-icon')
        for icon in icons:
            label = icon.find('span')
            icon_class = icon.find('i')
            if label and icon_class:
                name = label.get_text(strip=True)
                is_checked = 'fa-check' in icon_class.get('class', [])
                if name in features_map:
                    apartment_dict[features_map[name]] = int(is_checked)

        # === שליפת מחיר מהכותרת ===
        price_tags = soup.select('h2.card-title')
        for tag in price_tags:
            text = tag.get_text(strip=True)
            if '₪' in text:
                apartment_dict['price'] = text.replace("₪", "").replace(",", "").strip()
                break
        
        # טיפול בערכים טקסטואליים מיוחדים
        if 'תאריך כניסה' in apartment_dict:
            if 'מיידית' in apartment_dict['תאריך כניסה']:
                apartment_dict['תאריך כניסה'] = '0'

        # === פיצול קומה וקומות בבניין ===
        if 'קומה' in apartment_dict:
            floor_raw = apartment_dict['קומה']
            if 'מתוך' in floor_raw:
                parts = floor_raw.split('מתוך')
                floor = parts[0].strip()
                total_floors = parts[1].strip()
                apartment_dict['floor'] = '0' if 'קרקע' in floor else floor
                apartment_dict['total_floors'] = '0' if 'קרקע' in total_floors else total_floors
            else:
                apartment_dict['floor'] = '0' if 'קרקע' in floor_raw else floor_raw.strip()
                apartment_dict['total_floors'] = None
        else:
            apartment_dict['floor'] = None
            apartment_dict['total_floors'] = None

        # שמירת הקישור
        apartment_dict['link'] = link

        return apartment_dict

    except Exception as e:
        print(f"שגיאה בפריט {link}: {e}")
        return None


In [6]:
all_apartments = []

for neighborhood, code in area_codes.items():
    print(f"\n⬇️ מתחיל גרידת שכונה: {neighborhood}")
    links = get_apartment_links_by_area(code, max_pages=2)

    for link in links:
        info = get_apartment_info(link)

        if not info:
            continue  # דילוג על מודעות ללא תוכן בכלל
            
        info["שכונה"] = neighborhood
        all_apartments.append(info)
        time.sleep(0.5)

print(f"\n📦 לאחר סינון: נשארו {len(all_apartments)} מודעות עם עיר = תל אביב ונתונים תקינים")




⬇️ מתחיל גרידת שכונה: הצפון הישן - החלק המרכזי
שגיאה בפריט https://www.ad.co.il/ad/15954517: name 'nan' is not defined
שגיאה בפריט https://www.ad.co.il/ad/15924513: name 'nan' is not defined

⬇️ מתחיל גרידת שכונה: התקווה
שגיאה בפריט https://www.ad.co.il/nadlanrent: name 'nan' is not defined

⬇️ מתחיל גרידת שכונה: בית שטראוס
שגיאה בפריט https://www.ad.co.il/ad/14436379: name 'nan' is not defined
שגיאה בפריט https://www.ad.co.il/nadlanrent: name 'nan' is not defined

📦 לאחר סינון: נשארו 107 מודעות עם עיר = תל אביב ונתונים תקינים


In [7]:
all_links = [item['link'] for item in all_apartments]
all_links

['https://www.ad.co.il/ad/16190954',
 'https://www.ad.co.il/ad/16145737',
 'https://www.ad.co.il/ad/16206752',
 'https://www.ad.co.il/ad/15953921',
 'https://www.ad.co.il/ad/15894434',
 'https://www.ad.co.il/ad/16200666',
 'https://www.ad.co.il/ad/16199712',
 'https://www.ad.co.il/ad/14173708',
 'https://www.ad.co.il/ad/16192780',
 'https://www.ad.co.il/ad/16191255',
 'https://www.ad.co.il/ad/16191426',
 'https://www.ad.co.il/ad/16191430',
 'https://www.ad.co.il/ad/16191029',
 'https://www.ad.co.il/ad/16191136',
 'https://www.ad.co.il/ad/16190753',
 'https://www.ad.co.il/ad/16189778',
 'https://www.ad.co.il/ad/16152772',
 'https://www.ad.co.il/ad/16088869',
 'https://www.ad.co.il/ad/16069609',
 'https://www.ad.co.il/ad/16067980',
 'https://www.ad.co.il/ad/16060191',
 'https://www.ad.co.il/ad/16060355',
 'https://www.ad.co.il/ad/16046915',
 'https://www.ad.co.il/ad/16028668',
 'https://www.ad.co.il/ad/16023116',
 'https://www.ad.co.il/ad/15695084',
 'https://www.ad.co.il/ad/15599541',
 

In [8]:
detailed_apartments = []

for link in all_links:
    print(f"🔍 שואב מידע מ: {link}")
    info = get_apartment_info(link)
    if info:
        detailed_apartments.append(info)
    time.sleep(0.5)  # בעדינות מול האתר וה-API

print(f"\n✅ הסתיים! נאספו {len(detailed_apartments)} דירות")

# המרה ל-DataFrame
import pandas as pd
df = pd.DataFrame(detailed_apartments)




🔍 שואב מידע מ: https://www.ad.co.il/ad/16190954
🔍 שואב מידע מ: https://www.ad.co.il/ad/16145737
🔍 שואב מידע מ: https://www.ad.co.il/ad/16206752
🔍 שואב מידע מ: https://www.ad.co.il/ad/15953921
🔍 שואב מידע מ: https://www.ad.co.il/ad/15894434
🔍 שואב מידע מ: https://www.ad.co.il/ad/16200666
🔍 שואב מידע מ: https://www.ad.co.il/ad/16199712
🔍 שואב מידע מ: https://www.ad.co.il/ad/14173708
🔍 שואב מידע מ: https://www.ad.co.il/ad/16192780
🔍 שואב מידע מ: https://www.ad.co.il/ad/16191255
🔍 שואב מידע מ: https://www.ad.co.il/ad/16191426
🔍 שואב מידע מ: https://www.ad.co.il/ad/16191430
🔍 שואב מידע מ: https://www.ad.co.il/ad/16191029
🔍 שואב מידע מ: https://www.ad.co.il/ad/16191136
🔍 שואב מידע מ: https://www.ad.co.il/ad/16190753
🔍 שואב מידע מ: https://www.ad.co.il/ad/16189778
🔍 שואב מידע מ: https://www.ad.co.il/ad/16152772
🔍 שואב מידע מ: https://www.ad.co.il/ad/16088869
🔍 שואב מידע מ: https://www.ad.co.il/ad/16069609
🔍 שואב מידע מ: https://www.ad.co.il/ad/16067980
🔍 שואב מידע מ: https://www.ad.co.il/ad/1

In [9]:
df = pd.DataFrame(detailed_apartments)



In [10]:
# רשימת סוגי נדל"ן שאנחנו רוצים להשאיר
valid_types = ["דירה", "דירת גן", "גג/פנטהאוז", "דופלקס", "יחידת דיור"]

# סינון הדאטה פריים
df = df[df['פרטי הנכס'].isin(valid_types)]
print(f"✅ נשארו {len(df)} מודעות נדל\"ן רלוונטיות")


✅ נשארו 88 מודעות נדל"ן רלוונטיות


In [11]:
# שומר רק עמודות שיש בהן לפחות ערך אחד לא-ריק
df = df.dropna(axis=1, how='all')
df.columns


Index(['פרטי הנכס', 'אזור', 'עיר', 'שכונה', 'כתובת', 'חדרים', 'מרפסות', 'קומה',
       'שטח בנוי', 'תאריך כניסה', 'תשלומים בשנה', 'ארנונה בחודש',
       'ועד בית בחודש', 'description', 'num_of_images', 'distance_from_center',
       'has_parking', 'has_stotsge', 'elevator', 'ac', 'handicap', 'has_bars',
       'has_safe_room', 'has_balcon', 'is_furnished', 'is_renovated', 'price',
       'floor', 'total_floors', 'link', 'שטח גינה'],
      dtype='object')

In [12]:
df = df[df['עיר'] == "תל אביב יפו"]
print(f"✅ לאחר סינון לפי עיר: נשארו {len(df)} דירות בתל אביב יפו בלבד")


✅ לאחר סינון לפי עיר: נשארו 88 דירות בתל אביב יפו בלבד


In [13]:
rename_columns = {
    'פרטי הנכס': 'property_type',
    'אזור': 'region',
    'עיר': 'city',
    'שכונה': 'neighborhood',
    'כתובת': 'address',
    'חדרים': 'room_num',
    'מרפסות': 'balconies',  #  לא דרוש
    'קומה': 'floor_original',  # נשמר זמנית
    'שטח בנוי': 'area',
    'שטח גינה': 'garden_area',
    'תאריך כניסה': 'days_to_enter',
    'תשלומים בשנה': 'num_of_payments',
    'ארנונה בחודש': 'monthly_arnona',
    'ועד בית בחודש': 'building_tax',
    'קומות בבניין': 'total_floors',
    'תיאור': 'description',
    'מחיר': 'price'
}

df = df.rename(columns=rename_columns)


In [14]:
# הסדר הסופי הרצוי של העמודות
new_order = [
    'property_type', 'neighborhood', 'address', 'room_num', 'floor', 
    'area', 'garden_area', 'days_to_enter', 'num_of_payments', 
    'monthly_arnona', 'building_tax', 'total_floors', 'description', 
    'has_parking', 'has_stotsge', 'elevator', 'ac', 'handicap', 
    'has_bars', 'has_safe_room', 'has_balcon', 'is_furnished', 
    'is_renovated', 'price', 'num_of_images', 'distance_from_center'
]

#  בדיקה אם חסרות עמודות
missing_columns = [col for col in new_order if col not in df.columns]
if missing_columns:
    print("❌ חסרות העמודות הבאות:")
    for col in missing_columns:
        print("-", col)
    raise ValueError("לא כל העמודות הדרושות קיימות. עצירה מיידית.")
else:
    print("✅ כל העמודות קיימות. אפשר להמשיך!")

# סידור לפי סדר העמודות
df = df[new_order]


✅ כל העמודות קיימות. אפשר להמשיך!


In [15]:
for col in df.columns:
    print(f"{col}: {df[col].dtype}")


property_type: object
neighborhood: object
address: object
room_num: object
floor: object
area: object
garden_area: object
days_to_enter: object
num_of_payments: object
monthly_arnona: object
building_tax: object
total_floors: object
description: object
has_parking: int64
has_stotsge: int64
elevator: int64
ac: int64
handicap: int64
has_bars: int64
has_safe_room: int64
has_balcon: int64
is_furnished: int64
is_renovated: int64
price: object
num_of_images: int64
distance_from_center: int64


In [16]:
strict_types = {
    'property_type': 'string',  
    'neighborhood': 'string',
    'address': 'string',
    'room_num': 'float64',
    'floor': 'Int64',
    'area': 'Int64',
    'garden_area': 'Int64',
    'days_to_enter': 'Int64',
    'num_of_payments': 'Int64',
    'monthly_arnona': 'Int64',
    'building_tax': 'Int64',
    'total_floors': 'Int64',
    'description': 'string',
    'has_parking': 'Int64',
    'has_stotsge': 'Int64',
    'elevator': 'Int64',
    'ac': 'Int64',
    'handicap': 'Int64',
    'has_bars': 'Int64',
    'has_safe_room': 'Int64',
    'has_balcon': 'Int64',
    'is_furnished': 'Int64',
    'is_renovated': 'Int64',
    'price': 'float64',
    'num_of_images': 'Int64',
    'distance_from_center': 'float64',
}

# המרה עם בדיקת עמודות קיימות
for col, dtype in strict_types.items():
    if col in df.columns:
        try:
            if 'float' in dtype or 'int' in dtype:
                df[col] = pd.to_numeric(df[col], errors='coerce').astype(dtype)
            else:
                df[col] = df[col].astype(dtype)
        except Exception as e:
            print(f"שגיאה בהמרה של {col} ל-{dtype}: {e}")
    else:
        print(f"⚠️ העמודה {col} לא קיימת ב-DataFrame")

# הצגה לבדיקה
print("✅ הטיפוסים לאחר המרה:")
print(df.dtypes)

✅ הטיפוסים לאחר המרה:
property_type            string
neighborhood             string
address                  string
room_num                float64
floor                     Int64
area                      Int64
garden_area               Int64
days_to_enter             Int64
num_of_payments           Int64
monthly_arnona            Int64
building_tax              Int64
total_floors              Int64
description              string
has_parking               Int64
has_stotsge               Int64
elevator                  Int64
ac                        Int64
handicap                  Int64
has_bars                  Int64
has_safe_room             Int64
has_balcon                Int64
is_furnished              Int64
is_renovated              Int64
price                   float64
num_of_images             Int64
distance_from_center    float64
dtype: object
